# AquaGuard AI Search Algorithms Notebook

This notebook gathers all six graph-search algorithms used for routing groundwater locations on the road network:
1. Breadth-First Search (BFS)
2. Depth-First Search (DFS)
3. Greedy Best-First Search (GBFS)
4. A* Search (A*)
5. Iterative Deepening DFS (IDDFS)
6. Bidirectional Search


## Setup – load road graph and snapped groundwater locations

We load the Pune road network (directed MultiDiGraph) and the CSV that contains the matched groundwater points.  Only rows with `match_status == 'Matched'` are kept.


In [5]:
import os
import pandas as pd
import networkx as nx
import osmnx as ox
import time
import heapq
import math

# Import helper functions from bfs_search (already implemented)
from bfs_search import bfs_route, _load_graph, _load_snap_csv, _shortest_edge_length

# Load the full directed graph (projected to UTM – metres)
G = _load_graph()

# Load matched groundwater snap points
snap_df = _load_snap_csv()
print('Loaded graph with', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges')
print('Matched groundwater locations:', len(snap_df))

Loaded graph with 170253 nodes and 409399 edges
Matched groundwater locations: 117


## 1️⃣ Breadth‑First Search (BFS)

BFS explores the graph level‑by‑level and therefore finds a path that uses the **fewest number of edges** (ignoring edge lengths).  The implementation is taken directly from `bfs_search.py` and reused here.


In [6]:
# BFS synthetic test (same as in bfs_search)
def synthetic_bfs_test():
    G_test = nx.MultiDiGraph()
    G_test.add_edge(1, 2, length=10)
    G_test.add_edge(1, 3, length=5)
    G_test.add_edge(1, 3, length=3)  # parallel, shorter edge
    G_test.add_edge(2, 4, length=7)
    G_test.add_edge(3, 4, length=12)
    res = bfs_route(start_node=1, target_node=4, G=G_test)
    print('Synthetic BFS test result:', res)

synthetic_bfs_test()

Synthetic BFS test result: {'success': True, 'route': [1, 2, 4], 'distance_m': 17.0, 'expanded_nodes': 4, 'edge_count': 2, 'time_s': 0.00012826919555664062}


### Real‑world demonstration on the Pune graph (BFS)

We pick the first two matched groundwater points as source and destination and run BFS.


In [7]:
if len(snap_df) >= 2:
    src = int(snap_df.iloc[0]['nearest_node_id'])
    dst = int(snap_df.iloc[1]['nearest_node_id'])
    print(f'Running BFS from node {src} to node {dst} …')
    result = bfs_route(start_node=src, target_node=dst, G=G)
    print('BFS result:', result)
else:
    print('Not enough matched locations for a demo.')

Running BFS from node 12788595845 to node 12788595830 …
BFS result: {'success': True, 'route': [12788595845, 12788595847, 12788595856, 12788595830], 'distance_m': 237.65127855509024, 'expanded_nodes': 13, 'edge_count': 3, 'time_s': 0.00012612342834472656}


## 2️⃣ Depth‑First Search (DFS)

Depth‑First Search (DFS) explores a graph by going as deep as possible along each branch before backtracking. It uses a **stack** (LIFO) to store the frontier. DFS does **not** guarantee the shortest‑edge path, but it respects one‑way road directions.


In [8]:
def dfs_route(start_node: int, target_node: int, G: nx.MultiDiGraph = None) -> dict:
    """Depth‑First Search on a directed MultiDiGraph.

    Parameters
    ----------
    start_node : int
        Starting road‑node ID.
    target_node : int
        Destination road‑node ID.
    G : nx.MultiDiGraph, optional
        Graph to search; if ``None`` the global ``G`` is used.

    Returns
    -------
    dict
        Same structure as ``bfs_route``.
    """
    if G is None:
        G = globals().get('G')
    result = {
        'success': False,
        'route': None,
        'distance_m': None,
        'expanded_nodes': 0,
        'edge_count': None,
        'time_s': None,
    }
    if start_node not in G or target_node not in G:
        result['time_s'] = 0.0
        return result
    start_time = time.time()
    stack = [start_node]
    came_from = {start_node: None}
    expanded = 0
    while stack:
        current = stack.pop()
        expanded += 1
        if current == target_node:
            break
        for neighbor in G.successors(current):
            if neighbor not in came_from:
                stack.append(neighbor)
                came_from[neighbor] = current
    if target_node in came_from:
        path = []
        node = target_node
        while node is not None:
            path.append(node)
            node = came_from[node]
        path.reverse()
        total_dist = sum(_shortest_edge_length(G, u, v) for u, v in zip(path[:-1], path[1:]))
        result.update({
            'success': True,
            'route': path,
            'distance_m': total_dist,
            'expanded_nodes': expanded,
            'edge_count': len(path) - 1,
            'time_s': time.time() - start_time,
        })
    else:
        result.update({
            'expanded_nodes': expanded,
            'time_s': time.time() - start_time,
        })
    return result

### Synthetic DFS test

A tiny graph is built to verify that DFS returns a valid path and computes distances using the shortest parallel edge.


In [9]:
def synthetic_dfs_test():
    G_test = nx.MultiDiGraph()
    G_test.add_edge(1, 2, length=10)
    G_test.add_edge(1, 3, length=5)
    G_test.add_edge(1, 3, length=3)
    G_test.add_edge(2, 4, length=7)
    G_test.add_edge(3, 4, length=12)
    res = dfs_route(1, 4, G_test)
    print('Synthetic DFS test result:', res)

synthetic_dfs_test()

Synthetic DFS test result: {'success': True, 'route': [1, 3, 4], 'distance_m': 15, 'expanded_nodes': 3, 'edge_count': 2, 'time_s': 6.437301635742188e-05}


### Real‑world DFS demonstration on the Pune graph

We again use the first two matched groundwater nodes as source and destination.


In [10]:
if len(snap_df) >= 2:
    src = int(snap_df.iloc[0]['nearest_node_id'])
    dst = int(snap_df.iloc[1]['nearest_node_id'])
    print(f'Running DFS from node {src} to node {dst} …')
    dfs_res = dfs_route(src, dst, G)
    print('DFS result:', dfs_res)
else:
    print('Not enough matched locations for a demo.')

Running DFS from node 12788595845 to node 12788595830 …
DFS result: {'success': True, 'route': [12788595845, 5487410540, 4333660709, 3135054820, 2329122656, 4345786238, 4345786240, 4345786236, 4345786241, 6158870529, 12860474755, 5487410211, 5489645437, 5491931568, 5491931592, 13521719779, 12509147393, 13697928921, 5489645788, 5566828045, 5489572053, 5489572025, 5566826498, 12788333676, 12788333656, 12788431010, 5489572212, 5489572223, 5487410684, 5487410760, 4434222443, 13697546742, 13697546767, 13697546777, 13697648904, 13521777650, 4345817907, 4345817935, 4345817937, 4345817933, 4345817930, 4345817899, 13521764918, 13521764922, 13521750970, 13697399225, 12509147396, 13645096220, 5491932893, 12788639054, 5487410524, 5491932827, 12788595867, 12788595830], 'distance_m': 37021.62105717624, 'expanded_nodes': 62, 'edge_count': 53, 'time_s': 0.00036263465881347656}


## 3️⃣ Greedy Best‑First Search (GBFS)

Greedy Best‑First Search expands the node that appears **closest** to the goal according to a heuristic.  Here we use straight‑line (Euclidean) distance in metres between a node’s coordinates (``x``/``y`` stored on the graph) and the destination node.  The algorithm respects the directed, one‑way nature of the Pune road network but **does not** guarantee the shortest‑distance route.


In [11]:
def greedy_best_first_route(start_node: int, target_node: int, G: nx.MultiDiGraph = None) -> dict:
    """Greedy Best‑First Search on a directed MultiDiGraph.

    The heuristic is Euclidean distance (metres) between a node and the target.
    """
    if G is None:
        G = globals().get('G')
    result = {
        'success': False,
        'route': None,
        'distance_m': None,
        'expanded_nodes': 0,
        'edge_count': None,
        'time_s': None,
    }
    if start_node not in G or target_node not in G:
        result['time_s'] = 0.0
        return result
    target_x = G.nodes[target_node].get('x')
    target_y = G.nodes[target_node].get('y')
    if target_x is None or target_y is None:
        result['time_s'] = 0.0
        return result
    def heuristic(node):
        nx_val = G.nodes[node].get('x')
        ny_val = G.nodes[node].get('y')
        return math.hypot(nx_val - target_x, ny_val - target_y) if nx_val is not None else float('inf')
    start_time = time.time()
    frontier = []
    heapq.heappush(frontier, (heuristic(start_node), start_node))
    came_from = {start_node: None}
    expanded = 0
    while frontier:
        _, current = heapq.heappop(frontier)
        expanded += 1
        if current == target_node:
            break
        for neighbor in G.successors(current):
            if neighbor not in came_from:
                heapq.heappush(frontier, (heuristic(neighbor), neighbor))
                came_from[neighbor] = current
    if target_node in came_from:
        path = []
        node = target_node
        while node is not None:
            path.append(node)
            node = came_from[node]
        path.reverse()
        total_dist = sum(_shortest_edge_length(G, u, v) for u, v in zip(path[:-1], path[1:]))
        result.update({
            'success': True,
            'route': path,
            'distance_m': total_dist,
            'expanded_nodes': expanded,
            'edge_count': len(path) - 1,
            'time_s': time.time() - start_time,
        })
    else:
        result.update({
            'expanded_nodes': expanded,
            'time_s': time.time() - start_time,
        })
    return result

### Synthetic Greedy Best‑First Search test

We build a tiny graph with explicit ``x``/``y`` coordinates so the heuristic works.


In [12]:
def synthetic_gbfs_test():
    G_test = nx.MultiDiGraph()
    # Add nodes with coordinates (metres)
    G_test.add_node(1, x=0, y=0)
    G_test.add_node(2, x=10, y=0)
    G_test.add_node(3, x=0, y=10)
    G_test.add_node(4, x=10, y=10)
    # Edges (parallel edges possible)
    G_test.add_edge(1, 2, length=10)
    G_test.add_edge(1, 3, length=5)
    G_test.add_edge(1, 3, length=3)  # shorter parallel edge
    G_test.add_edge(2, 4, length=7)
    G_test.add_edge(3, 4, length=12)
    res = greedy_best_first_route(1, 4, G_test)
    print('Synthetic GBFS test result:', res)

synthetic_gbfs_test()

Synthetic GBFS test result: {'success': True, 'route': [1, 2, 4], 'distance_m': 17, 'expanded_nodes': 3, 'edge_count': 2, 'time_s': 4.57763671875e-05}


### Real‑world Greedy Best‑First Search demonstration on the Pune graph

We again use the first two matched groundwater nodes as source and destination.


In [13]:
if len(snap_df) >= 2:
    src = int(snap_df.iloc[0]['nearest_node_id'])
    dst = int(snap_df.iloc[1]['nearest_node_id'])
    print(f'Running Greedy Best‑First Search from node {src} to node {dst} …')
    gbfs_res = greedy_best_first_route(src, dst, G)
    print('Greedy Best‑First Search result:', gbfs_res)
else:
    print('Not enough matched locations for a demo.')

Running Greedy Best‑First Search from node 12788595845 to node 12788595830 …
Greedy Best‑First Search result: {'success': True, 'route': [12788595845, 12788595847, 12788595856, 12788595830], 'distance_m': 237.65127855509024, 'expanded_nodes': 4, 'edge_count': 3, 'time_s': 5.602836608886719e-05}


## 4️⃣ A* Search (A*)

A* combines the accumulated road distance (g(n)) with a straight‑line heuristic (h(n)) to guide the search toward the goal while guaranteeing optimality when the heuristic is admissible.  Here we use the Euclidean distance between projected node coordinates as h(n) (a lower bound on actual road distance).  The algorithm respects one‑way edges and chooses the shortest parallel edge when calculating g‑costs.


In [14]:
def astar_route(start_node: int, target_node: int, G: nx.MultiDiGraph = None) -> dict:
    """A* Search on a directed MultiDiGraph.

    g(n) – accumulated road distance (metres) using the shortest parallel edge.
    h(n) – Euclidean straight‑line distance (metres) between node and target.
    f(n) = g(n) + h(n) is used as the priority.
    """
    if G is None:
        G = globals().get('G')
    result = {
        'success': False,
        'route': None,
        'distance_m': None,
        'expanded_nodes': 0,
        'edge_count': None,
        'time_s': None,
    }
    if start_node not in G or target_node not in G:
        result['time_s'] = 0.0
        return result
    # Target coordinates (projected, metres)
    tx = G.nodes[target_node].get('x')
    ty = G.nodes[target_node].get('y')
    if tx is None or ty is None:
        result['time_s'] = 0.0
        return result
    def heuristic(n):
        nx_val = G.nodes[n].get('x')
        ny_val = G.nodes[n].get('y')
        return math.hypot(nx_val - tx, ny_val - ty) if nx_val is not None else float('inf')
    start_time = time.time()
    # Priority queue of (f, g, node)
    frontier = []
    heapq.heappush(frontier, (heuristic(start_node), 0.0, start_node))
    came_from = {start_node: None}
    g_score = {start_node: 0.0}
    expanded = 0
    while frontier:
        f_current, g_current, current = heapq.heappop(frontier)
        expanded += 1
        if current == target_node:
            break
        for neighbor in G.successors(current):
            # Edge cost – shortest parallel edge length
            edge_len = _shortest_edge_length(G, current, neighbor)
            tentative_g = g_current + edge_len
            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g
                f_neighbor = tentative_g + heuristic(neighbor)
                heapq.heappush(frontier, (f_neighbor, tentative_g, neighbor))
                came_from[neighbor] = current
    if target_node in came_from:
        # Reconstruct path
        path = []
        node = target_node
        while node is not None:
            path.append(node)
            node = came_from[node]
        path.reverse()
        total_dist = g_score[target_node]
        result.update({
            'success': True,
            'route': path,
            'distance_m': total_dist,
            'expanded_nodes': expanded,
            'edge_count': len(path) - 1,
            'time_s': time.time() - start_time,
        })
    else:
        result.update({
            'expanded_nodes': expanded,
            'time_s': time.time() - start_time,
        })
    return result

### Synthetic A* test – minimum‑distance route

We create a tiny graph with node coordinates so the Euclidean heuristic is admissible and verify that A* finds the true shortest‑distance path.


In [15]:
def synthetic_astar_test():
    G_test = nx.MultiDiGraph()
    # Nodes with coordinates (metres)
    G_test.add_node(1, x=0, y=0)
    G_test.add_node(2, x=5, y=0)
    G_test.add_node(3, x=0, y=5)
    G_test.add_node(4, x=10, y=0)
    # Edges – note different lengths
    G_test.add_edge(1, 2, length=5)   # direct short edge
    G_test.add_edge(2, 4, length=5)   # total 10 metres via 2
    G_test.add_edge(1, 3, length=8)   # longer edge
    G_test.add_edge(3, 4, length=8)   # total 16 via 3
    res = astar_route(1, 4, G_test)
    print('Synthetic A* test result:', res)

synthetic_astar_test()

Synthetic A* test result: {'success': True, 'route': [1, 2, 4], 'distance_m': 10.0, 'expanded_nodes': 3, 'edge_count': 2, 'time_s': 6.175041198730469e-05}


### Real‑world A* demonstration on the Pune graph

We again use the first two matched groundwater nodes as source and destination.


In [16]:
if len(snap_df) >= 2:
    src = int(snap_df.iloc[0]['nearest_node_id'])
    dst = int(snap_df.iloc[1]['nearest_node_id'])
    print(f'Running A* Search from node {src} to node {dst} …')
    astar_res = astar_route(src, dst, G)
    print('A* Search result:', astar_res)
else:
    print('Not enough matched locations for a demo.')

Running A* Search from node 12788595845 to node 12788595830 …
A* Search result: {'success': True, 'route': [12788595845, 12788595847, 5489645800, 5487410304, 12788595856, 12788595830], 'distance_m': 227.0555952882313, 'expanded_nodes': 12, 'edge_count': 5, 'time_s': 0.00011920928955078125}


### 5️⃣ Iterative Deepening DFS (IDDFS)

**Explanation**
Iterative Deepening Depth‑First Search (IDDFS) combines the space efficiency of Depth‑First Search with the optimality (in terms of number of edges) of Breadth‑First Search. It repeatedly runs a depth‑limited DFS (DLS) with increasing depth limits until the goal is found or a maximum depth is reached. Each iteration expands nodes up to the current depth limit.

- The algorithm expands nodes in a DFS order, but the depth limit ensures that the shallowest solution (fewest edges) is found first.
- It does **not** guarantee the shortest road distance, only the minimum‑edge path within the depth bound.
- The total number of expanded nodes is the sum of nodes expanded in each depth‑limited iteration.


In [17]:
import time

def _dls(start, target, depth, G):
    """Depth‑limited DFS. Returns (found, path, nodes_expanded)."""
    if depth < 0:
        return False, [], 0
    stack = [(start, [start])]
    visited = set([start])
    nodes_expanded = 0
    while stack:
        node, path = stack.pop()
        nodes_expanded += 1
        if node == target:
            return True, path, nodes_expanded
        if len(path) - 1 == depth:
            continue
        for succ in G.successors(node):
            if succ not in visited:
                visited.add(succ)
                stack.append((succ, path + [succ]))
    return False, [], nodes_expanded

def iddfs_route(start_node, target_node, G, max_depth=20):
    """Iterative Deepening DFS route finder.
    Returns dict with keys: success, route, distance_m, expanded_nodes, edge_count, time_s.
    """
    if start_node not in G or target_node not in G:
        return {
            'success': False,
            'route': [],
            'distance_m': 0.0,
            'expanded_nodes': 0,
            'edge_count': 0,
            'time_s': 0.0,
            'error': 'Invalid start or target node'
        }
    start_time = time.time()
    total_expanded = 0
    for depth in range(max_depth + 1):
        found, path, expanded = _dls(start_node, target_node, depth, G)
        total_expanded += expanded
        if found:
            distance = 0.0
            for u, v in zip(path[:-1], path[1:]):
                distance += _shortest_edge_length(G, u, v)
            return {
                'success': True,
                'route': path,
                'distance_m': distance,
                'expanded_nodes': total_expanded,
                'edge_count': len(path) - 1,
                'time_s': time.time() - start_time,
            }
    return {
        'success': False,
        'route': [],
        'distance_m': 0.0,
        'expanded_nodes': total_expanded,
        'edge_count': 0,
        'time_s': time.time() - start_time,
        'error': f'No path found within depth {max_depth}'
    }


In [18]:
# Synthetic graph test for IDDFS
import networkx as nx
G_test = nx.DiGraph()
G_test.add_edges_from([(1, 2), (2, 3), (3, 4), (1, 5), (5, 4)])
for u, v in G_test.edges():
    G_test[u][v]['length'] = 1
def _shortest_edge_length(G, u, v):
    data = G.get_edge_data(u, v)
    if data is None:
        return float('inf')
    if isinstance(data, dict) and any(isinstance(k, int) for k in data):
        return min(d.get('length', float('inf')) for d in data.values())
    return data.get('length', float('inf'))
res = iddfs_route(1, 4, G_test, max_depth=3)
print('Synthetic IDDFS result:', res)


Synthetic IDDFS result: {'success': True, 'route': [1, 5, 4], 'distance_m': 2.0, 'expanded_nodes': 7, 'edge_count': 2, 'time_s': 5.6743621826171875e-05}


In [19]:
if len(snap_df) >= 2:
    src = int(snap_df.iloc[0]['nearest_node_id'])
    dst = int(snap_df.iloc[1]['nearest_node_id'])
    print(f'Running IDDFS from {src} to {dst} (max_depth=30)')
    real_res = iddfs_route(src, dst, G, max_depth=30)
    print('Real‑world IDDFS result:', real_res)
else:
    print('Not enough matched locations for real‑world demo.')


Running IDDFS from 12788595845 to 12788595830 (max_depth=30)
Real‑world IDDFS result: {'success': True, 'route': [12788595845, 5487410527, 12788595844, 12788595830], 'distance_m': 402.04489699768334, 'expanded_nodes': 28, 'edge_count': 3, 'time_s': 4.363059997558594e-05}


## 6️⃣ Bidirectional Search

### What is Bidirectional Search?
Bidirectional Search runs two simultaneous searches:
1. **Forward Search:** Starts at the `start_node` and searches forward along outgoing directed edges (`G.successors`).
2. **Backward Search:** Starts at the `target_node` and searches backward along incoming directed edges (`G.predecessors`).

Instead of exploring a large single search ball of radius $d$ with complexity $O(b^d)$, bidirectional search explores two smaller search balls of radius $d/2$, reducing the time and space complexity to $O(b^{d/2})$. The searches stop as soon as their frontiers or visited sets intersect at a common **meeting node**.

### Key Principles & Requirements:
- **Directionality & One-Way Restrictions:** On a directed graph $G$, the forward search travels from node $u \to v$ (following $G.successors(u)$). The backward search moves in the reverse direction of the traffic flow, which means traveling from $w \to current$ (following $G.predecessors(current)$). This strictly guarantees that the resulting path only traverses roads in their legal, permissible directions.
- **Completeness & Optimality:** When both directions expand level-by-level (Breadth-First order) on unweighted/uniform-step graphs, bidirectional search is **complete** (guaranteed to find a route if one exists) and finds the path with the **minimum number of edges**.
- **Road Distance Calculation:** As with all our routing algorithms, the actual road distance in metres is calculated along the reconstructed route using the shortest parallel edge `length` between consecutive nodes via `_shortest_edge_length`.


In [20]:
from collections import deque
import time
import networkx as nx

def bidirectional_route(start_node: int, target_node: int, G: nx.MultiDiGraph = None) -> dict:
    """Bidirectional Breadth-First Search on a directed MultiDiGraph.
    
    Parameters
    ----------
    start_node : int
        Starting road-node ID.
    target_node : int
        Destination road-node ID.
    G : nx.MultiDiGraph, optional
        Directed road network. Defaults to global G.
        
    Returns
    -------
    dict
        Standard result dictionary with keys:
        'success', 'route', 'distance_m', 'expanded_nodes', 'edge_count', 'time_s'
    """
    if G is None:
        G = globals().get('G')
        
    result = {
        'success': False,
        'route': None,
        'distance_m': None,
        'expanded_nodes': 0,
        'edge_count': None,
        'time_s': None,
    }
    
    if G is None or start_node not in G or target_node not in G:
        result['time_s'] = 0.0
        return result
        
    start_time = time.time()
    
    # Trivial case: start is already destination
    if start_node == target_node:
        result.update({
            'success': True,
            'route': [start_node],
            'distance_m': 0.0,
            'expanded_nodes': 0,
            'edge_count': 0,
            'time_s': time.time() - start_time,
        })
        return result

    # Forward and backward queues
    q_f = deque([start_node])
    q_b = deque([target_node])
    
    # came_from dictionaries
    # forward: came_from_f[n] = predecessor of n in forward tree
    # backward: came_from_b[n] = successor of n in forward direction (parent in backward tree)
    came_from_f = {start_node: None}
    came_from_b = {target_node: None}
    
    expanded = 0
    meeting_node = None
    
    while q_f and q_b:
        # Expand one level from forward search
        level_size_f = len(q_f)
        for _ in range(level_size_f):
            current_f = q_f.popleft()
            expanded += 1
            
            for succ in G.successors(current_f):
                if succ not in came_from_f:
                    came_from_f[succ] = current_f
                    if succ in came_from_b:
                        meeting_node = succ
                        break
                    q_f.append(succ)
            if meeting_node is not None:
                break
                
        if meeting_node is not None:
            break
            
        # Expand one level from backward search (traversing predecessors to respect one-way edges)
        level_size_b = len(q_b)
        for _ in range(level_size_b):
            current_b = q_b.popleft()
            expanded += 1
            
            for pred in G.predecessors(current_b):
                if pred not in came_from_b:
                    came_from_b[pred] = current_b
                    if pred in came_from_f:
                        meeting_node = pred
                        break
                    q_b.append(pred)
            if meeting_node is not None:
                break
                
        if meeting_node is not None:
            break

    if meeting_node is not None:
        # Reconstruct path from start to meeting_node
        path_f = []
        curr = meeting_node
        while curr is not None:
            path_f.append(curr)
            curr = came_from_f[curr]
        path_f.reverse()
        
        # Reconstruct path from meeting_node to target_node
        path_b = []
        curr = came_from_b[meeting_node]
        while curr is not None:
            path_b.append(curr)
            curr = came_from_b[curr]
            
        complete_route = path_f + path_b
        
        # Calculate total road distance using shortest parallel edge length
        total_dist = sum(
            _shortest_edge_length(G, u, v)
            for u, v in zip(complete_route[:-1], complete_route[1:])
        )
        
        result.update({
            'success': True,
            'route': complete_route,
            'distance_m': total_dist,
            'expanded_nodes': expanded,
            'edge_count': len(complete_route) - 1,
            'time_s': time.time() - start_time,
        })
    else:
        result.update({
            'expanded_nodes': expanded,
            'time_s': time.time() - start_time,
        })
        
    return result


### Synthetic Bidirectional Search Test

We build a small directed test graph with parallel edges and one-way constraints to verify:
1. Forward and backward search meet correctly.
2. One-way directionality is strictly obeyed (backward search follows predecessors).
3. Shortest parallel edge lengths are selected for distance calculation.


In [21]:
def synthetic_bidirectional_test():
    G_test = nx.MultiDiGraph()
    # Edges: 1 -> 2 -> 4 and 1 -> 3 -> 4
    # Parallel edge between 1 and 2: lengths 10 and 6
    G_test.add_edge(1, 2, length=10)
    G_test.add_edge(1, 2, length=6)   # Shorter parallel edge
    G_test.add_edge(2, 4, length=7)
    G_test.add_edge(1, 3, length=5)
    G_test.add_edge(3, 4, length=12)
    
    res = bidirectional_route(1, 4, G_test)
    print('Synthetic Bidirectional Search result:', res)

synthetic_bidirectional_test()


Synthetic Bidirectional Search result: {'success': True, 'route': [1, 2, 4], 'distance_m': 13, 'expanded_nodes': 2, 'edge_count': 2, 'time_s': 9.655952453613281e-05}


### Real-world Bidirectional Search demonstration on Pune road network

We run Bidirectional Search between the first two matched groundwater locations from `pune_location_node_snap.csv` on the full Pune directed road graph.


In [22]:
if len(snap_df) >= 2:
    src = int(snap_df.iloc[0]['nearest_node_id'])
    dst = int(snap_df.iloc[1]['nearest_node_id'])
    print(f'Running Bidirectional Search from node {src} to node {dst} …')
    bidi_res = bidirectional_route(src, dst, G)
    print('Bidirectional Search result:', bidi_res)
else:
    print('Not enough matched locations for a demo.')


Running Bidirectional Search from node 12788595845 to node 12788595830 …
Bidirectional Search result: {'success': True, 'route': [12788595845, 12788595847, 12788595856, 12788595830], 'distance_m': 237.65127855509024, 'expanded_nodes': 3, 'edge_count': 3, 'time_s': 6.508827209472656e-05}
